In [0]:
ext_loc='abfss://stagingcontainer@venu72sa.dfs.core.windows.net'
csv_file=ext_loc+'/bronze/csv/Scenario_1.csv'
json_file=ext_loc+'/bronze/json/Listings.json'  
bathworkflow =ext_loc+'/bronze/csv/bathworkflow.csv'

In [0]:
mapping_df = spark.read.option("header", True).format('csv').option('header','true').load(bathworkflow)
display(mapping_df)

In [0]:
from pyspark.sql.functions import lower

mapping_df = mapping_df.withColumn("number", lower(col("word"))) \
                       .withColumn("value", col("value").cast("int"))
mapping_df.display()

In [0]:
df=spark.read.format('csv').option('header','true').load(csv_file)
display(df)

In [0]:
#df.schema

In [0]:
from pyspark.sql.types import *
listings_schema=StructType([StructField('MLS #', StringType(), True),
             StructField('Bedrooms', StringType(), True), 
             StructField('Full Baths', StringType(), True), 
             StructField('Half Baths', StringType(), True), 
             StructField('Access/Detailed Show Inst', StringType(), True)
             ])

In [0]:
df=spark.read.format('csv').option('header','true')\
.schema(listings_schema).load(csv_file)
display(df)

In [0]:
#RENAME COLUMNS
df = df.withColumnRenamed("MLS #", "mls_id")\
       .withColumnRenamed("Bedrooms", "bedrooms")\
       .withColumnRenamed("Full Baths", "full_baths")\
       .withColumnRenamed("Half Baths", "half_baths")\
       .withColumnRenamed("Access/Detailed Show Inst", "access_instructions")

In [0]:
df.display()

> Conver the string data to integer/flot for 3 columns , i.e bedrooms , full_baths, half_baths 

In [0]:
df = df.withColumn("bedrooms_lower", lower(col("bedrooms")))

df = df.alias("a").join(
    mapping_df.alias("b"),
    col("a.bedrooms_lower") == col("b.word"),
    "left"
).select(
    "a.*",
    col("b.value").alias("bedrooms_num_mapped")
    #col("c.value").alias("full_baths_num_mapped")
)
df.display()

In [0]:
from pyspark.sql.functions import col, when, lower, regexp_replace, split

# ----------------------------
# 1. RENAME COLUMNS
# ----------------------------
df = df.withColumnRenamed("MLS #", "mls_id") \
       .withColumnRenamed("Bedrooms", "bedrooms") \
       .withColumnRenamed("Full Baths", "full_baths") \
       .withColumnRenamed("Half Baths", "half_baths") \
       .withColumnRenamed("Access/Detailed Show Inst", "access_instructions")

# ----------------------------
# 2. MAP TEXT → NUMERIC
# ----------------------------
mapping_expr = {
    "none": 0,
    "one": 1,
    "two": 2,
    "three": 3,
    "four": 4,
    "five": 5,
    "ten+": 10
}

def map_to_number(column):
    expr = None
    for k, v in mapping_expr.items():
        cond = when(lower(col(column)) == k, v)
        expr = cond if expr is None else expr.when(lower(col(column)) == k, v)
    return expr.otherwise(None)

df = df.withColumn("bedrooms_num", map_to_number("bedrooms")) \
       .withColumn("full_baths_num", map_to_number("full_baths")) \
       .withColumn("half_baths_num", map_to_number("half_baths"))

# ----------------------------
# 3. CREATE TOTALS
# ----------------------------
df = df.withColumn(
    "bedroomTotal",
    col("bedrooms_num")
).withColumn(
    "bathroomTotal",
    col("full_baths_num") + (col("half_baths_num") * 0.5)
)

# ----------------------------
# 4. CLEAN ACCESS INSTRUCTIONS
# ----------------------------
df = df.withColumn(
    "access_clean",
    lower(col("access_instructions"))
)

# standardize text
df = df.withColumn(
    "access_clean",
    regexp_replace(col("access_clean"), "la", "listing agent")
)

# ----------------------------
# 5. CONVERT TO ARRAY (COLLECTION TYPE)
# ----------------------------
df = df.withColumn(
    "access_instructions_array",
    split(col("access_clean"), ",")
)

# ----------------------------
# 6. FINAL RESULT
# ----------------------------
ds=df.select(
    "mls_id","full_baths","full_baths_num","half_baths","half_baths_num","bedrooms","bedrooms_num",    
    "bedroomTotal",
    "bathroomTotal",
    "access_instructions_array"
)
ds.display()
